## 1. FAISS

Facebook Al Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

##Loading Text from speech.txt
loader = TextLoader("/home/prashant/Documents/Remove and Reinstall NVIDIA.txt")
docs = loader.load()

## Chunking the text into smaller pieces
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks = text_splitter.split_documents(docs)
chunks

[Document(metadata={'source': '/home/prashant/Documents/Remove and Reinstall NVIDIA.txt'}, page_content="Remove and Reinstall NVIDIA Drivers\n\nTo remove and reinstall the latest NVIDIA driver on Ubuntu, follow these steps:\n\n1.  **Remove Existing Drivers:** Open a terminal and run the following commands to completely purge all existing NVIDIA packages and their dependencies:\n    ```bash\n    sudo apt-get purge --autoremove '^nvidia-.*'\n    ```\n    This command removes all packages starting with 'nvidia-' and cleans up any unused dependencies  For a more thorough removal, especially if drivers were installed via the `.run` file, you may also need to run:\n    ```bash\n    sudo apt purge libnvidia-*\n    ```\n    \n\n2.  **Disable Secure Boot (if necessary):** If you encounter issues during installation, particularly with newer drivers, ensure Secure Boot is disabled. You can check its status with:\n    ```bash\n    sudo mokutil --sb-state\n    ```\n    If it is enabled, you will ne

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, embeddings)
db

In [9]:
query="What are the steps to remove the existing drivers?"

## getting the relevant documents for the query
docs_res = db.similarity_search(query)
docs_res[2].page_content

"Remove and Reinstall NVIDIA Drivers\n\nTo remove and reinstall the latest NVIDIA driver on Ubuntu, follow these steps:\n\n1.  **Remove Existing Drivers:** Open a terminal and run the following commands to completely purge all existing NVIDIA packages and their dependencies:\n    ```bash\n    sudo apt-get purge --autoremove '^nvidia-.*'\n    ```\n    This command removes all packages starting with 'nvidia-' and cleans up any unused dependencies  For a more thorough removal, especially if drivers were installed via the `.run` file, you may also need to run:\n    ```bash\n    sudo apt purge libnvidia-*\n    ```\n    \n\n2.  **Disable Secure Boot (if necessary):** If you encounter issues during installation, particularly with newer drivers, ensure Secure Boot is disabled. You can check its status with:\n    ```bash\n    sudo mokutil --sb-state\n    ```\n    If it is enabled, you will need to disable it in your system's BIOS/UEFI settings"

#### As a Retriever

We can also convert the vectorestore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retriever.

In [10]:
retriever=db.as_retriever()
retriever.invoke(query)

[Document(id='b430f72f-6f59-4180-932f-e4cc43cbae0a', metadata={'source': '/home/prashant/Documents/Remove and Reinstall NVIDIA.txt'}, page_content='3.  **Add the Graphics Drivers PPA:** To ensure access to the latest available drivers, add the official Ubuntu graphics drivers Personal Package Archive (PPA):\n    ```bash\n    sudo add-apt-repository ppa:graphics-drivers/ppa\n    ```\n    After adding the PPA, update your package list:\n    ```bash\n    sudo apt update\n    ```\n\n4.  **Install the Latest Driver:** You can install the latest recommended driver automatically by running:\n    ```bash\n    sudo ubuntu-drivers autoinstall\n    ```\n    This command detects your GPU and installs the most suitable driver version available in the PPA  Alternatively, you can install a specific version by replacing `XX` with the desired version number (e.g., `560`):\n    ```bash\n    sudo apt install nvidia-driver-XX\n    ```\n    Check the [PPA page](https://launchpad.net/~graphics-drivers/+arch

#### Similarity Search with score

There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [11]:
doc_and_score=db.similarity_search_with_score(query)
doc_and_score

[(Document(id='b430f72f-6f59-4180-932f-e4cc43cbae0a', metadata={'source': '/home/prashant/Documents/Remove and Reinstall NVIDIA.txt'}, page_content='3.  **Add the Graphics Drivers PPA:** To ensure access to the latest available drivers, add the official Ubuntu graphics drivers Personal Package Archive (PPA):\n    ```bash\n    sudo add-apt-repository ppa:graphics-drivers/ppa\n    ```\n    After adding the PPA, update your package list:\n    ```bash\n    sudo apt update\n    ```\n\n4.  **Install the Latest Driver:** You can install the latest recommended driver automatically by running:\n    ```bash\n    sudo ubuntu-drivers autoinstall\n    ```\n    This command detects your GPU and installs the most suitable driver version available in the PPA  Alternatively, you can install a specific version by replacing `XX` with the desired version number (e.g., `560`):\n    ```bash\n    sudo apt install nvidia-driver-XX\n    ```\n    Check the [PPA page](https://launchpad.net/~graphics-drivers/+arc

Generating a Embedding Vector of the query and running the similarity search on generated Vector.

In [12]:
query_vector = embeddings.embed_query(query)
query_vector

[0.006634690333157778,
 0.06868281215429306,
 0.028775524348020554,
 -0.005691936239600182,
 0.0008429261506535113,
 -0.01599222421646118,
 -0.007256901822984219,
 -0.05280473828315735,
 -0.08787474036216736,
 -0.06092514097690582,
 0.07826054841279984,
 0.006389525253325701,
 -0.0056676981039345264,
 -0.045117612928152084,
 -0.06297910958528519,
 0.047755587846040726,
 -0.03954954072833061,
 -0.0066907890141010284,
 0.05680650472640991,
 0.03385158255696297,
 -0.05621503293514252,
 -0.09328684210777283,
 -0.10467088967561722,
 0.030052542686462402,
 0.029041852802038193,
 -0.007899720221757889,
 0.11293547600507736,
 0.07771763205528259,
 0.01198719535022974,
 -0.01478953380137682,
 -0.015905052423477173,
 0.05451738461852074,
 0.10651986300945282,
 -0.03701544925570488,
 -0.01875501684844494,
 -0.14681458473205566,
 0.06049327179789543,
 -0.021968899294734,
 -0.0193929560482502,
 -0.13980498909950256,
 0.017777426168322563,
 -0.03264646977186203,
 -0.02545343153178692,
 -0.0307844746

In [14]:
doc_score=db.similarity_search_by_vector(query_vector)
doc_score

[Document(id='b430f72f-6f59-4180-932f-e4cc43cbae0a', metadata={'source': '/home/prashant/Documents/Remove and Reinstall NVIDIA.txt'}, page_content='3.  **Add the Graphics Drivers PPA:** To ensure access to the latest available drivers, add the official Ubuntu graphics drivers Personal Package Archive (PPA):\n    ```bash\n    sudo add-apt-repository ppa:graphics-drivers/ppa\n    ```\n    After adding the PPA, update your package list:\n    ```bash\n    sudo apt update\n    ```\n\n4.  **Install the Latest Driver:** You can install the latest recommended driver automatically by running:\n    ```bash\n    sudo ubuntu-drivers autoinstall\n    ```\n    This command detects your GPU and installs the most suitable driver version available in the PPA  Alternatively, you can install a specific version by replacing `XX` with the desired version number (e.g., `560`):\n    ```bash\n    sudo apt install nvidia-driver-XX\n    ```\n    Check the [PPA page](https://launchpad.net/~graphics-drivers/+arch

In [15]:
## Saving the vector database
db.save_local("faiss_index")

In [18]:
vec_df = FAISS.load_local("faiss_index", embeddings,allow_dangerous_deserialization=True)
vec_df.similarity_search(query)

[Document(id='b430f72f-6f59-4180-932f-e4cc43cbae0a', metadata={'source': '/home/prashant/Documents/Remove and Reinstall NVIDIA.txt'}, page_content='3.  **Add the Graphics Drivers PPA:** To ensure access to the latest available drivers, add the official Ubuntu graphics drivers Personal Package Archive (PPA):\n    ```bash\n    sudo add-apt-repository ppa:graphics-drivers/ppa\n    ```\n    After adding the PPA, update your package list:\n    ```bash\n    sudo apt update\n    ```\n\n4.  **Install the Latest Driver:** You can install the latest recommended driver automatically by running:\n    ```bash\n    sudo ubuntu-drivers autoinstall\n    ```\n    This command detects your GPU and installs the most suitable driver version available in the PPA  Alternatively, you can install a specific version by replacing `XX` with the desired version number (e.g., `560`):\n    ```bash\n    sudo apt install nvidia-driver-XX\n    ```\n    Check the [PPA page](https://launchpad.net/~graphics-drivers/+arch

## 2.Chroma

Chroma is a AI-native open-source vector database focused on developer productivity and happiness.Chroma is licensed under Apache 2.0.

In [22]:
from langchain_community.vectorstores import Chroma

# using the earlier generated documents chunks
db_chroma = Chroma.from_documents(chunks, embeddings,persist_directory="./chroma_index")
db_chroma

In [23]:
db_chroma.similarity_search(query)

[Document(metadata={'source': '/home/prashant/Documents/Remove and Reinstall NVIDIA.txt'}, page_content='3.  **Add the Graphics Drivers PPA:** To ensure access to the latest available drivers, add the official Ubuntu graphics drivers Personal Package Archive (PPA):\n    ```bash\n    sudo add-apt-repository ppa:graphics-drivers/ppa\n    ```\n    After adding the PPA, update your package list:\n    ```bash\n    sudo apt update\n    ```\n\n4.  **Install the Latest Driver:** You can install the latest recommended driver automatically by running:\n    ```bash\n    sudo ubuntu-drivers autoinstall\n    ```\n    This command detects your GPU and installs the most suitable driver version available in the PPA  Alternatively, you can install a specific version by replacing `XX` with the desired version number (e.g., `560`):\n    ```bash\n    sudo apt install nvidia-driver-XX\n    ```\n    Check the [PPA page](https://launchpad.net/~graphics-drivers/+archive/ubuntu/ppa) to find the latest availabl

In [26]:
get_db = Chroma(persist_directory="./chroma_index", embedding_function=embeddings)
get_db

In [27]:
get_db.similarity_search(query)

[Document(metadata={'source': '/home/prashant/Documents/Remove and Reinstall NVIDIA.txt'}, page_content='3.  **Add the Graphics Drivers PPA:** To ensure access to the latest available drivers, add the official Ubuntu graphics drivers Personal Package Archive (PPA):\n    ```bash\n    sudo add-apt-repository ppa:graphics-drivers/ppa\n    ```\n    After adding the PPA, update your package list:\n    ```bash\n    sudo apt update\n    ```\n\n4.  **Install the Latest Driver:** You can install the latest recommended driver automatically by running:\n    ```bash\n    sudo ubuntu-drivers autoinstall\n    ```\n    This command detects your GPU and installs the most suitable driver version available in the PPA  Alternatively, you can install a specific version by replacing `XX` with the desired version number (e.g., `560`):\n    ```bash\n    sudo apt install nvidia-driver-XX\n    ```\n    Check the [PPA page](https://launchpad.net/~graphics-drivers/+archive/ubuntu/ppa) to find the latest availabl

In [28]:
## As Retriever
retriever_chroma = db_chroma.as_retriever()
retriever_chroma.invoke(query)

[Document(metadata={'source': '/home/prashant/Documents/Remove and Reinstall NVIDIA.txt'}, page_content='3.  **Add the Graphics Drivers PPA:** To ensure access to the latest available drivers, add the official Ubuntu graphics drivers Personal Package Archive (PPA):\n    ```bash\n    sudo add-apt-repository ppa:graphics-drivers/ppa\n    ```\n    After adding the PPA, update your package list:\n    ```bash\n    sudo apt update\n    ```\n\n4.  **Install the Latest Driver:** You can install the latest recommended driver automatically by running:\n    ```bash\n    sudo ubuntu-drivers autoinstall\n    ```\n    This command detects your GPU and installs the most suitable driver version available in the PPA  Alternatively, you can install a specific version by replacing `XX` with the desired version number (e.g., `560`):\n    ```bash\n    sudo apt install nvidia-driver-XX\n    ```\n    Check the [PPA page](https://launchpad.net/~graphics-drivers/+archive/ubuntu/ppa) to find the latest availabl